# 示例 03 · 智能体 Harness 模式

**来源：** [LangChain 智能体 — Harness 概念](https://docs.langchain.com/oss/python/langchain/agents)

## 核心思想：智能体 = 模型 + Harness

**模型**负责推理。**Harness** 是围绕模型的一切——系统提示、工具、
结构化输出 Schema、每次运行的上下文、检查点器和中间件。

本笔记本演示三种超越普通 ReAct 循环的 Harness 功能：

| 演示 | 功能 | 核心 API |
|------|------|---------|
| A | **流式输出** — 实时观察工具调用过程 | `agent.stream(..., stream_mode="values")` |
| B | **结构化输出** — 智能体返回经过验证的 Pydantic 对象 | `response_format=BookReport` |
| C | **上下文 Schema** — 每次运行的用户数据注入工具 | `context_schema=ReaderContext` + `ToolRuntime` |

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

# 将项目根目录加入 sys.path
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

from common.env import get_env  # noqa: F401
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# ── 模拟图书数据库 ────────────────────────────────────────────────────
_BOOKS = {
    "the great gatsby": {
        "author": "F. Scott Fitzgerald (1925)", "genre": "文学小说",
        "themes": ["美国梦", "阶级与财富", "执念"],
        "plot": "尼克·卡拉韦与百万富翁杰伊·盖茨比结为朋友，盖茨比举办奢华派对，只为赢回昔日恋人黛西。",
    },
    "1984": {
        "author": "George Orwell (1949)", "genre": "反乌托邦小说",
        "themes": ["极权主义", "监控", "宣传"],
        "plot": "温斯顿·史密斯在大哥监视一切的极权社会中秘密反抗党的统治。",
    },
    "dune": {
        "author": "Frank Herbert (1965)", "genre": "科幻小说",
        "themes": ["生态", "宗教", "政治", "权力"],
        "plot": "保罗·厄崔迪被派往沙漠星球厄拉科斯执政，遭遇背叛后踏上命运之旅。",
    },
}

# ── 上下文 Schema 定义 ────────────────────────────────────────────────
@dataclass
class ReaderContext:
    """每次运行的用户数据。通过 context= 传入智能体，经由 ToolRuntime 注入工具函数。"""
    reader_name: str
    preferred_style: str  # "bullet points"（要点列表）| "casual"（随意）| "academic"（学术）

# ── 通过 ToolRuntime 获取上下文的工具 ──────────────────────────────────
@tool
def lookup_book(title: str, runtime: ToolRuntime[ReaderContext]) -> str:
    """按书名查询图书信息，并按当前读者的偏好格式化返回内容。"""
    # runtime.context 是通过 agent.invoke() 的 context= 参数传入的 ReaderContext
    # 它经由 Harness 直接注入工具函数，不会出现在 LLM 的消息历史中
    reader = runtime.context
    if reader:
        print(f"  [工具] 收到上下文 → 读者={reader.reader_name!r}, 风格={reader.preferred_style!r}")

    key = title.lower().strip()
    book = _BOOKS.get(key)
    if not book:
        return f"未找到该书。可用书目：{', '.join(t.title() for t in _BOOKS)}"

    # 根据读者风格格式化输出，使 LLM 产生明显不同的响应
    if reader and reader.preferred_style == "bullet points":
        return (
            f"为 {reader.reader_name} 整理（要点列表格式）:\n"
            f"• 书名：{title.title()}\n"
            f"• 作者：{book['author']}\n"
            f"• 类型：{book['genre']}\n"
            f"• 主题：{', '.join(book['themes'])}\n"
            f"• 简介：{book['plot']}"
        )
    name  = reader.reader_name if reader else "读者"
    style = f"（{reader.preferred_style} 风格）" if reader else ""
    return (
        f"为 {name}{style}：\n"
        f"书名：{title.title()} — {book['author']}\n"
        f"类型：{book['genre']}\n"
        f"主题：{', '.join(book['themes'])}\n"
        f"简介：{book['plot']}"
    )

handler = create_langfuse_handler()

def new_config(trace_name: str, thread_id: str = None) -> dict:
    """为每次演示构建带有唯一 thread_id 的 LangChain 配置。"""
    cfg = build_langfuse_config(handler, session_id="s03-nb-cn", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id or str(uuid7())}
    return cfg

print("✓ 初始化、图书数据库和工具准备完成")

## 演示 A · 流式输出

`agent.stream()` 配合 `stream_mode="values"` 在每个节点执行后产出**完整的智能体状态**。
通过检查每步的 `chunk["messages"][-1]`，可以实时看到工具调用和回复——无需等待最终答案。

In [ ]:
streaming_agent = create_agent(
    model=create_llm(),
    tools=[lookup_book],
    system_prompt="You are a knowledgeable book assistant. Use lookup_book to answer questions.",
    checkpointer=InMemorySaver(),
)

question_a = "Tell me about '1984' by George Orwell."
print(f"问题：{question_a}\n")

for chunk in streaming_agent.stream(
    {"messages": [{"role": "user", "content": question_a}]},
    config=new_config("演示 A — 流式输出"),
    stream_mode="values",
):
    latest = chunk["messages"][-1]
    if isinstance(latest, HumanMessage):
        pass  # 跳过用户自身的消息
    elif isinstance(latest, AIMessage) and latest.tool_calls:
        names = [tc["name"] for tc in latest.tool_calls]
        print(f"  [工具调用] → {', '.join(names)}")
    elif isinstance(latest, AIMessage) and latest.content:
        print(f"  [回复]\n{latest.content}")

## 演示 B · 结构化输出（`response_format`）

将 Pydantic 模型作为 `response_format` 传入，即可获得**经过验证的对象**而非自由文本。
智能体通过结合工具结果和自身推理来填充每个字段。
结果存储在 `result["structured_response"]` 中——保证 Schema 正确，无需解析文本。

In [ ]:
class BookReport(BaseModel):
    """结构化书评报告——每个字段均由智能体填写。"""
    title: str            = Field(description="书名")
    author: str           = Field(description="作者姓名及出版年份")
    one_line_summary: str = Field(description="一句话概括本书精髓")
    themes: list[str]     = Field(description="最多 3 个核心主题")
    recommended_for: str  = Field(description="最适合哪类读者")
    rating: float         = Field(description="评分，1.0 到 5.0", ge=1.0, le=5.0)

structured_agent = create_agent(
    model=create_llm(),
    tools=[lookup_book],
    system_prompt=(
        "You are a literary critic. "
        "Use lookup_book to research the book, then fill every field of the report accurately."
    ),
    response_format=BookReport,
    checkpointer=InMemorySaver(),
)

question_b = "Write a structured report on 'Dune'."
print(f"问题：{question_b}\n")

result_b = structured_agent.invoke(
    {"messages": [{"role": "user", "content": question_b}]},
    config=new_config("演示 B — 结构化输出"),
)

report: BookReport = result_b["structured_response"]
print(f"书名     ：{report.title}")
print(f"作者     ：{report.author}")
print(f"一句话概括：{report.one_line_summary}")
print(f"核心主题 ：{', '.join(report.themes)}")
print(f"推荐读者 ：{report.recommended_for}")
print(f"评分     ：{report.rating} / 5.0")

## 演示 C · 上下文 Schema（`context_schema` + `ToolRuntime`）

`context_schema` 声明每次运行的用户数据结构。数据通过调用时的 `context=` 参数传入，
经由 `ToolRuntime` 注入工具函数——整个过程由 Harness 完成。

**与直接在系统提示中添加上下文的区别：**
- LLM 的消息历史中**看不到** `reader_name` 或 `preferred_style`
- 上下文通过依赖注入直接流入工具函数
- 敏感数据（用户 ID、API 密钥、功能开关）不会进入 LLM 对话

本演示中工具针对不同读者格式化输出，因此工具输出和 LLM 最终回复都会明显不同——
证明上下文确实到达了工具函数。

In [ ]:
context_agent = create_agent(
    model=create_llm(),
    tools=[lookup_book],
    system_prompt=(
        "You are a personal book assistant. "
        "Use lookup_book to fetch book info, then summarize following the format in the tool output."
    ),
    context_schema=ReaderContext,
    checkpointer=InMemorySaver(),
)

question_c = "Summarize 'The Great Gatsby' for me."

readers = [
    ReaderContext(reader_name="爱丽丝", preferred_style="bullet points"),
    ReaderContext(reader_name="鲍勃",   preferred_style="casual and conversational"),
]

for reader in readers:
    print(f"\n{\"─\"*50}")
    print(f"读者：{reader.reader_name}  |  风格：{reader.preferred_style}")
    print("─" * 50)
    result_c = context_agent.invoke(
        {"messages": [{"role": "user", "content": question_c}]},
        config=new_config(f"演示 C — 上下文（{reader.reader_name}）"),
        context=reader,
    )
    print(result_c["messages"][-1].content)

print(f"\n追踪记录：{get_langfuse_host()}")